In [1]:
import os
import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(1)

In [2]:
import sys
import tensorflow as tf
import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Devices:", tf.config.list_physical_devices())

Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
TensorFlow: 2.21.0
NumPy: 2.5.2
Pandas: 3.0.5
Scikit-learn: 1.9.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


# File Paths


In [3]:
from pathlib import Path

MODEL_PATH = Path("../models/LSTM_HPO.keras")
DATA_PATH = Path("../data/fdia_dataset_processed.npz")
RESULTS_DIR = Path("../results")

print("Model exists:", MODEL_PATH.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Results folder exists:", RESULTS_DIR.exists())

print("\nModel:", MODEL_PATH.resolve())
print("Dataset:", DATA_PATH.resolve())
print("Results:", RESULTS_DIR.resolve())

Model exists: True
Dataset exists: True
Results folder exists: True

Model: C:\tinyml-fdia-windows\models\LSTM_HPO.keras
Dataset: C:\tinyml-fdia-windows\data\fdia_dataset_processed.npz
Results: C:\tinyml-fdia-windows\results


# Dataset

In [4]:
import numpy as np

data = np.load(DATA_PATH)

X_test = data["X_test"]
y_test = data["y_test"]

X_test_lstm = X_test.transpose(0, 2, 1).astype(np.float32)

print("X_test:", X_test.shape)
print("X_test_lstm:", X_test_lstm.shape)
print("y_test:", y_test.shape)

X_test: (9720, 6, 83)
X_test_lstm: (9720, 83, 6)
y_test: (9720,)


In [5]:
np.random.seed(42)

sample_size = int(0.10 * X_test_lstm.shape[0])

timing_idx = np.random.choice(
    X_test_lstm.shape[0],
    sample_size,
    replace=False
)

X_timing = X_test_lstm[timing_idx]

print("Total test samples:", len(X_test_lstm))
print("Timing samples:", len(X_timing))
print("Timing shape:", X_timing.shape)

Total test samples: 9720
Timing samples: 972
Timing shape: (972, 83, 6)


In [6]:
import gc
import timeit
import pandas as pd

from sklearn.metrics import accuracy_score, recall_score

def measure_inference_time(model, X_timing):
    times = []

    for sample in X_timing:
        sample = np.expand_dims(sample, axis=0)

        start = timeit.default_timer()

        model.predict(sample, verbose=0)

        end = timeit.default_timer()

        times.append((end - start) * 1000)

        gc.collect()

    return np.array(times)

# LSTM Originl test

In [7]:
print("Testing: LSTM_Original")

model = tf.keras.models.load_model(MODEL_PATH)

# Full test set evaluation
y_prob = model.predict(X_test_lstm, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Inference timing
times = measure_inference_time(model, X_timing)

mean_time = np.mean(times)

result = pd.DataFrame([{
    "model": "LSTM_Original",
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "inference_time_ms": mean_time
}])

print("\n=== FINAL RESULT ===")
print(result.to_string(index=False))

Testing: LSTM_Original

=== FINAL RESULT ===
        model  accuracy  fdia_recall  fault_recall  inference_time_ms
LSTM_Original  0.989198     0.987847      0.990415          97.964871


# MLP Original Test

In [8]:
MLP_MODEL_PATH = Path("../models/MLP_HPO.keras")

X_timing_mlp = X_test[timing_idx].astype(np.float32)

print("Model exists:", MLP_MODEL_PATH.exists())
print("Total test samples:", len(X_test))
print("Timing samples:", len(X_timing_mlp))
print("Timing shape:", X_timing_mlp.shape)

Model exists: True
Total test samples: 9720
Timing samples: 972
Timing shape: (972, 6, 83)


In [9]:
def measure_mlp_time(model, X_timing):
    times = []

    for sample in X_timing:
        sample = np.expand_dims(sample, axis=0)

        start = timeit.default_timer()

        model.predict(sample, verbose=0)

        end = timeit.default_timer()

        times.append((end - start) * 1000)

        gc.collect()

    return np.array(times)

In [10]:
print("Testing: MLP_Original")

model = tf.keras.models.load_model(MLP_MODEL_PATH)

# Full test set evaluation
y_prob = model.predict(X_test.astype(np.float32), verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Inference timing
times_mlp = measure_mlp_time(
    model,
    X_timing_mlp
)

mean_time = np.mean(times_mlp)

result_mlp = pd.DataFrame([{
    "model": "MLP_Original",
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "inference_time_ms": mean_time
}])

print("\n=== FINAL RESULT ===")
print(result_mlp.to_string(index=False))

Testing: MLP_Original

=== FINAL RESULT ===
       model  accuracy  fdia_recall  fault_recall  inference_time_ms
MLP_Original  0.998354     0.998264      0.998435          96.974322


In [19]:
import os
import tensorflow as tf

print("CPU count:", os.cpu_count())

print(
    "Intra-op threads:",
    tf.config.threading.get_intra_op_parallelism_threads()
)

print(
    "Inter-op threads:",
    tf.config.threading.get_inter_op_parallelism_threads()
)

CPU count: 16
Intra-op threads: 0
Inter-op threads: 0
